In [ ]:
!pip install Augmentor grad_cam

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models.vision_transformer import vit_b_16
from torch.cuda.amp import autocast, GradScaler
import numpy as np
from PIL import Image
import os, shutil, random, time, warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import Augmentor
import wandb
from pytorch_grad_cam import GradCAM
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Constants & Model Config

In [ ]:
labels = ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash',
          'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
num_classes = len(labels)

RAW_PATH = '/kaggle/input/datasets/djihadelhakbelkhir/realwaste/RealWaste/'
PROCESSED_BASE = '/kaggle/working/RealWaste_processed'
TRAIN_PATH = os.path.join(PROCESSED_BASE, 'train')
VAL_PATH   = os.path.join(PROCESSED_BASE, 'val')
TEST_PATH  = os.path.join(PROCESSED_BASE, 'test')

MODEL_CONFIG = {
    'vit':       {'size': 224, 'freeze_points': [10, 5, 0]},
    'densenet':  {'size': 224, 'freeze_points': [143, 80, 0]},
    'inception': {'size': 299, 'freeze_points': [249, 120, 0]},
    'mobilenet': {'size': 224, 'freeze_points': [149, 70, 0]},
    'vgg':       {'size': 224, 'freeze_points': [11, 5, 0]},
    'resnet':    {'size': 224, 'freeze_points': [140, 80, 0]},
}

BATCH_SIZE_BASE = 32
EPOCHS_PER_STAGE = 5

## Preprocessing + Augmentation 

In [ ]:
def preprocess_and_augment():
    for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
        os.makedirs(p, exist_ok=True)
        for lbl in labels:
            os.makedirs(os.path.join(p, lbl), exist_ok=True)

    print("=== 1. Splitting 80/10/10 ===")
    for label in tqdm(labels):
        src = os.path.join(RAW_PATH, label)
        files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.png','.jpeg'))]
        random.shuffle(files)
        n = len(files)
        train_n = int(0.8 * n)
        val_n   = int(0.1 * n)
        for i, f in enumerate(files):
            src_file = os.path.join(src, f)
            if i < train_n:
                dst = os.path.join(TRAIN_PATH, label, f"train_{label}_{i}.jpg")
            elif i < train_n + val_n:
                dst = os.path.join(VAL_PATH, label, f"val_{label}_{i}.jpg")
            else:
                dst = os.path.join(TEST_PATH, label, f"test_{label}_{i}.jpg")
            shutil.copy(src_file, dst)

    # RGB fix
    def fix_rgb(folder):
        for label in labels:
            path = os.path.join(folder, label)
            for f in tqdm(os.listdir(path), desc=f"RGB {label}"):
                with Image.open(os.path.join(path, f)) as im:
                    if im.mode != 'RGB':
                        im = im.convert('RGB')
                    im.save(os.path.join(path, f))

    print("=== 2. RGB Fix ===")
    fix_rgb(TRAIN_PATH); fix_rgb(VAL_PATH); fix_rgb(TEST_PATH)

    # Geometric augmentation with Augmentor (exactly what you asked for)
    print("=== 3. Geometric Augmentation (Augmentor) ===")
    def augment_folder(folder, label, multiplier=3):
        p = Augmentor.Pipeline(os.path.join(folder, label))
        p.flip_left_right(probability=1.0)
        p.random_distortion(probability=1.0, grid_width=5, grid_height=5, magnitude=4)
        p.rotate(probability=1.0, max_left_rotation=25, max_right_rotation=25)
        p.shear(probability=1.0, max_shear_left=15, max_shear_right=15)
        p.sample(multiplier * len(os.listdir(os.path.join(folder, label))))

    for label in labels:
        augment_folder(TRAIN_PATH, label, multiplier=3)
        augment_folder(VAL_PATH,   label, multiplier=2)

    print("✅ Preprocessing + Augmentation complete!")

preprocess_and_augment()

## DataLoaders (model-specific size)

In [ ]:
def get_dataloaders(model_name):
    size = MODEL_CONFIG[model_name]['size']
    bs = 20 if size == 299 else BATCH_SIZE_BASE

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.15),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.3),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize(320 if size==299 else 256),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])

    train_ds = datasets.ImageFolder(TRAIN_PATH, train_tf)
    val_ds   = datasets.ImageFolder(VAL_PATH,   val_tf)
    test_ds  = datasets.ImageFolder(TEST_PATH,  val_tf)

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)

    # === CRITICAL FIX: force float32 weights ===
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, lbl in train_ds:
        counts[lbl] += 1
    class_weights = torch.tensor(1.0 / (counts + 1e-6), dtype=torch.float32).to(DEVICE)

    print(f"✅ {model_name} loaders ready | batch={bs} | weights=float32")
    return train_loader, val_loader, test_loader, class_weights

## model factory

In [ ]:
def get_model(model_name):
    cfg = MODEL_CONFIG[model_name]
    
    if model_name == 'vit':
        m = vit_b_16(weights='IMAGENET1K_V1')
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
        
    elif model_name == 'densenet':
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        
    elif model_name == 'inception':
        m = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)  # ← FIXED: no aux_logits=False
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        # aux_logits remains True (required for pretrained weights)
        
    elif model_name == 'mobilenet':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        
    elif model_name == 'vgg':
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(4096, num_classes)
        
    elif model_name == 'resnet':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        
    return m.to(DEVICE)

## train utils

In [ ]:
scaler = GradScaler() if torch.cuda.is_available() else None

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_total, preds, trues = 0.0, [], []
    for x, y in tqdm(loader, desc="Train", leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE).long()                    # ← forces correct dtype
        optimizer.zero_grad()
        with autocast():
            out = model(x)
            if isinstance(out, tuple): out = out[0]  # Inception aux
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_total += loss.item()
        preds.extend(out.argmax(1).cpu().numpy())
        trues.extend(y.cpu().numpy())
    f1 = precision_recall_fscore_support(trues, preds, average='weighted')[2]
    return loss_total / len(loader), f1

def validate_epoch(model, loader, criterion):
    model.eval()
    loss_total, preds, trues = 0.0, [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Val", leave=False):
            x = x.to(DEVICE)
            y = y.to(DEVICE).long()                # ← forces correct dtype
            out = model(x)
            if isinstance(out, tuple): out = out[0]
            loss = criterion(out, y)
            loss_total += loss.item()
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(y.cpu().numpy())
    f1 = precision_recall_fscore_support(trues, preds, average='weighted')[2]
    return loss_total / len(loader), f1

## training funciton

In [ ]:
import time

def train_model(model_name, lr=1e-4, wd=1e-2):
    print(f"🚀 Starting training: {model_name.upper()} | LR={lr} | WD={wd}")
    
    model = get_model(model_name)
    train_loader, val_loader, _, class_weights = get_dataloaders(model_name)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    freeze_points = MODEL_CONFIG[model_name]['freeze_points']
    best_f1 = 0.0
    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}  # for later plots

    total_start = time.time()

    for stage in range(3):
        stage_start = time.time()
        print(f"\n{'='*15} STAGE {stage+1}/3 — Unfreezing from layer {freeze_points[stage]} {'='*15}")
        
        # Unfreeze layers
        for i, param in enumerate(model.parameters()):
            param.requires_grad = (i >= freeze_points[stage])
        
        for epoch in range(EPOCHS_PER_STAGE):
            epoch_start = time.time()
            
            t_loss, t_f1 = train_epoch(model, train_loader, criterion, optimizer)
            v_loss, v_f1 = validate_epoch(model, val_loader, criterion)
            scheduler.step(v_f1)
            
            current_lr = optimizer.param_groups[0]['lr']
            
            # Store history
            history['train_loss'].append(t_loss)
            history['train_f1'].append(t_f1)
            history['val_loss'].append(v_loss)
            history['val_f1'].append(v_f1)
            
            # Print rich status
            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch+1:02d}/{EPOCHS_PER_STAGE} | "
                  f"Train Loss: {t_loss:.4f} | Train F1: {t_f1:.4f} | "
                  f"Val Loss: {v_loss:.4f} | Val F1: {v_f1:.4f} | "
                  f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")
            
            # Update best
            if v_f1 > best_f1:
                best_f1 = v_f1
                torch.save(model.state_dict(), f"best_{model_name}.pth")
                print(f"   → New best Val F1: {best_f1:.4f} (saved checkpoint)")

        # Stage summary
        stage_time = time.time() - stage_start
        print(f"✅ STAGE {stage+1} COMPLETE | Best F1 so far: {best_f1:.4f} | Stage time: {stage_time/60:.1f} min\n")

    total_time = time.time() - total_start
    print(f"🎉 {model_name.upper()} TRAINING FINISHED!")
    print(f"   Final Best Val F1: {best_f1:.4f}")
    print(f"   Total training time: {total_time/60:.1f} minutes")
    print(f"   Best model saved as: best_{model_name}.pth\n")

    # Optional: save history for later plotting
    torch.save(history, f"{model_name}_history.pth")
    
    return best_f1

## test and evaluation

In [ ]:
import pandas as pd

def evaluate_on_test(model_name):
    print(f"\n🔍 FINAL TEST EVALUATION — {model_name.upper()}")
    
    model = get_model(model_name)
    model.load_state_dict(torch.load(f"best_{model_name}.pth", weights_only=True))
    model.eval()
    
    _, _, test_loader, _ = get_dataloaders(model_name)
    
    preds, trues = [], []
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Testing"):
            x = x.to(DEVICE)
            y = y.to(DEVICE).long()
            out = model(x)
            if isinstance(out, tuple): out = out[0]
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(y.cpu().numpy())
    
    # Classification report
    report = classification_report(trues, preds, target_names=labels, digits=4, output_dict=True)
    df_report = pd.DataFrame(report).round(4).T
    
    # Overall metrics
    test_acc = (np.array(preds) == np.array(trues)).mean()
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']
    
    # === Save per-model detailed CSV (for reference) ===
    df_report['model'] = model_name
    df_report.to_csv(f"test_metrics_{model_name}.csv")
    
    # === Add to master summary CSV (updates after every model) ===
    summary_row = {
        'model': model_name,
        'test_accuracy': round(test_acc, 4),
        'macro_f1': round(macro_f1, 4),
        'weighted_f1': round(weighted_f1, 4),
        'precision_macro': round(report['macro avg']['precision'], 4),
        'recall_macro': round(report['macro avg']['recall'], 4)
    }
    
    summary_df = pd.DataFrame([summary_row])
    
    csv_path = "test_results_summary.csv"
    if os.path.exists(csv_path):
        summary_df.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        summary_df.to_csv(csv_path, index=False)
    
    # === Confusion Matrix ===
    cm = confusion_matrix(trues, preds)
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix — {model_name} (Test Set)')
    plt.xticks(rotation=45, ha='right')
    plt.savefig(f"confusion_matrix_{model_name}.png")
    plt.close()
    
    print(f"✅ Test Accuracy: {test_acc:.4f} | Macro F1: {macro_f1:.4f}")
    print(f"📁 Saved: test_metrics_{model_name}.csv + confusion_matrix_{model_name}.png")
    print(f"📊 Master summary CSV updated → test_results_summary.csv")
    
    return test_acc, macro_f1

** model training , evaluation and comparaision

In [ ]:
models_to_try = [  'mobilenet', 'vgg','inception','vit', 'densenet',]
results = {}
test_results = {}

for m in models_to_try:
    print(f"\n{'='*40} TRAINING {m.upper()} {'='*40}")
    best_f1 = train_model(m, lr=1e-4)
    results[m] = best_f1
    
    # ← Test evaluation + CSV + visualization
    test_acc, macro_f1 = evaluate_on_test(m)
    test_results[m] = {'accuracy': test_acc, 'macro_f1': macro_f1}

# ===================== FINAL VISUALIZATION =====================
print("\n" + "="*70)
print("🎨 FINAL MODEL COMPARISON VISUALIZATION")
print("="*70)

summary_df = pd.read_csv("test_results_summary.csv")

plt.figure(figsize=(12, 6))
x = summary_df['model']
width = 0.35

plt.bar(x, summary_df['test_accuracy'], width, label='Test Accuracy', color='#1f77b4')
plt.bar([i + width for i in range(len(x))], summary_df['macro_f1'], width, label='Macro F1-Score', color='#ff7f0e')

plt.title('Final Test Performance Comparison (All Models)', fontsize=16, pad=20)
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("model_comparison_chart.png")
plt.show()

print("📊 Comparison chart saved as: model_comparison_chart.png")
print("📁 All results saved in: test_results_summary.csv")

In [ ]:
!pip install onnxscript
!pip install tensorrt

## onnnx and latency of a model

In [ ]:
import time
import torch

def export_onnx_and_latency(model_name):
    model = get_model(model_name)
    model.load_state_dict(torch.load(f"best_{model_name}.pth", weights_only=True))
    model.eval()

    size = MODEL_CONFIG[model_name]['size']
    dummy = torch.randn(1, 3, size, size).to(DEVICE)

    torch.onnx.export(
    model,
    dummy,
    f"best_{model_name}.onnx",
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)

    # Warmup
    for _ in range(10):
        model(dummy)

    # Latency measurement
    start = time.time()
    for _ in range(100):
        model(dummy)
    end = time.time()

    avg_latency = (end - start) / 100 * 1000

    print(f"✅ ONNX exported: best_{model_name}.onnx | Avg latency: {avg_latency:.2f} ms")

    return f"best_{model_name}.onnx"


export_onnx_and_latency("mobilenet")


## Densenet Data loader test

In [ ]:
def test_pipeline():
    loader, _, _, _ = get_dataloaders('densenet')
    x, y = next(iter(loader))
    assert x.shape == (32, 3, 224, 224), "Batch/shape wrong"
    assert y.dtype == torch.int64, "Label dtype error"
    
    model = get_model('densenet')
    model.load_state_dict(torch.load("best_densenet.pth", weights_only=True))
    out = model(x.to(DEVICE))
    assert out.shape == (32, 9), "Output shape wrong"
    print("✅ Pipeline unit + integration test PASSED")

test_pipeline()